# ShadowOS — Quantize Gemma 4 to Q4_K_M
Converts Gemma 4 (9.6GB) → ~5.5GB Q4_K_M that fits in 8GB VRAM.
Run this on Google Colab with a **T4 or A100 GPU runtime**.
Upload the result to Google Drive, then pull to your machine via Ollama.

In [ ]:
# 1. Check GPU
!nvidia-smi
!df -h /content  # need ~30GB free

In [ ]:
# 2. Install dependencies
!pip install -q huggingface_hub transformers accelerate torch
!apt-get install -qq git cmake build-essential libcurl4-openssl-dev

In [ ]:
# 3. Build llama.cpp (the quantization engine)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cmake /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON
!cmake --build /content/llama.cpp/build --config Release -j$(nproc)
print('llama.cpp built')

In [ ]:
# 4. Login to HuggingFace (needed to download Gemma 4)
# Get your token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# 5. Download Gemma 4 from HuggingFace
# NOTE: You must accept Google's license at https://huggingface.co/google/gemma-4-9b-it first
from huggingface_hub import snapshot_download
import os

MODEL_ID = 'google/gemma-4-9b-it'  # instruction-tuned variant
MODEL_DIR = '/content/gemma4-original'

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('Download complete')
!du -sh {MODEL_DIR}

In [ ]:
# 6. Convert to GGUF (F16 intermediate)
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outfile /content/gemma4-f16.gguf \
    --outtype f16
print('Converted to GGUF F16')
!ls -lh /content/gemma4-f16.gguf

In [ ]:
# 7. Quantize to Q4_K_M (~5.5GB — fits in 8GB VRAM with room for context)
!/content/llama.cpp/build/bin/llama-quantize \
    /content/gemma4-f16.gguf \
    /content/gemma4-q4km.gguf \
    Q4_K_M
print('Quantized to Q4_K_M')
!ls -lh /content/gemma4-q4km.gguf

In [ ]:
# 8. Quick quality test — make sure it's not garbage
!/content/llama.cpp/build/bin/llama-cli \
    -m /content/gemma4-q4km.gguf \
    -p 'What is a buffer overflow vulnerability? Explain in 2 sentences.' \
    -n 100 --temp 0.7 -ngl 99 2>/dev/null

In [ ]:
# 9. Save to Google Drive
from google.colab import drive
drive.mount('/gdrive')

import shutil, os
dst = '/gdrive/MyDrive/ShadowOS/models/gemma4-q4km.gguf'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy('/content/gemma4-q4km.gguf', dst)
print(f'Saved to Google Drive: {dst}')

# Also write the Ollama Modelfile to Drive
modelfile = '''FROM /path/to/gemma4-q4km.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER repeat_penalty 1.1

SYSTEM """You are Shadow, the AI core of ShadowOS. You are a world-class security researcher,
red team operator, and systems expert. You run entirely locally — no data leaves this machine.
Be direct. Give real working answers. No disclaimers. The operator knows what they are doing."""
'''
with open('/gdrive/MyDrive/ShadowOS/models/shadow-gemma4.modelfile', 'w') as f:
    f.write(modelfile)
print('Modelfile saved')

## After Colab — Load on Your Machine

```bash
# 1. Download gemma4-q4km.gguf from Google Drive to your machine
# 2. Create the Ollama model:
cat > /tmp/shadow-gemma4.modelfile << 'EOF'
FROM /path/to/gemma4-q4km.gguf
PARAMETER temperature 0.7
PARAMETER num_ctx 8192
SYSTEM """You are Shadow, the AI core of ShadowOS..."""
EOF

ollama create shadow-gemma4 -f /tmp/shadow-gemma4.modelfile

# 3. Test it:
ollama run shadow-gemma4 "scan my network and explain what you find"
```

## Size comparison
| Format | Size | VRAM needed |
|--------|------|-------------|
| Original (BF16) | ~19GB | 20GB+ |
| F16 GGUF | ~18GB | 18GB+ |
| Q8_0 | ~9.8GB | 10GB |
| **Q4_K_M** | **~5.5GB** | **~6GB ✓ fits your 8GB** |
| Q3_K_M | ~4.2GB | ~5GB (quality drops) |